In [1]:
from pathlib import Path, PosixPath
import asyncio
import logging
import re
from urllib.parse import urlparse

import dask.bag as db
from dask.diagnostics import ProgressBar
from dask.distributed import Client, LocalCluster, WorkerPlugin, get_worker
from dask.distributed import as_completed
from coiled import Cluster as CoiledCluster

from obstore.store import S3Store
import rustac
import duckdb
import s3fs
import os

import dask
from dask.distributed import get_worker
from pathlib import Path, PosixPath
import logging


logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [2]:
class RustacStorePlugin(WorkerPlugin):
    def __init__(self, read_config, write_config):
        self.store_config = read_config
        self.write_store_config = write_config
        self.store = None
        self.writer_store = None
    
    def setup(self, worker):
        """Initialize the store when a worker starts"""
        bucket,prefix,region = self.store_config
        self.store = S3Store(bucket=bucket, prefix=prefix, region=region, client_options={"timeout":"8m"}, skip_signature=True)
        bucket,prefix,region = self.write_store_config
        self.writer_store = S3Store(bucket=bucket, prefix=prefix, region=region, client_options={"timeout":"10m"})
        print("Initialized rustac store")
    
    def teardown(self, worker):
        """Cleanup when worker shuts down"""
        if self.store:
            print("Cleaned up rustac store")
    
    def get_store(self):
        if self.store is None:
            raise RuntimeError("Store not initialized")
        return self.store


def create_dask_cluster(environment="local",
                        store=None,
                        n_workers=4,
                        threads_per_worker=1,
                        cloud_opts={}):

    if "client" in locals() and "cluster" in locals():
        return (client, cluster)
    else:
        if environment=="local":
            print("Creating new local Dask client")
            cluster = LocalCluster(n_workers=n_workers,
                                 threads_per_worker=threads_per_worker,
                                 silence_logs=logging.ERROR)
        else:
            print("Creating new Coiled Dask client")
            cluster = CoiledCluster(n_workers=n_workers,
                                    **cloud_opts)
            cluster.send_private_envs(
                {
                    "AWS_SECRET_ACCESS_KEY": os.os.getenv("AWS_SECRET_ACCESS_KEY"),
                    "AWS_ACCESS_KEY_ID": os.os.getenv("AWS_ACCESS_KEY_ID")
                }
            )
            source_store_opts = store["input"]
            output_store_opts = store["output"]
            obstore_plugin = RustacStorePlugin(source_store_opts, output_store_opts)
            client.register_plugin(obstore_plugin, name='rustac_store')
        client = Client(cluster)
        return (client, cluster)

In [3]:
source_store = (
    "its-live-data",
    "test-space/cloud-experiments/catalog/sentinel1-consolidated/",
    "us-west-2"
)

output_store = (
    "its-live-data",
    "test-space/cloud-experiments/catalog/geoparquet/sentinel1/",
    "us-west-2"
)

cloud_store = {
    "input": source_store,
    "output": output_store
}

local_store = None

cloud_opts = {
    "region": "us-west-2",
    "worker_vm_types": ["r5a.2xlarge"],
    "spot_policy":"on-demand",
}

client, cluster = create_dask_cluster(
    environment="local",
    store = local_store,
    n_workers=4,
    threads_per_worker=16,
    cloud_opts=cloud_opts)
client

Creating new local Dask client


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 64,Total memory: 46.75 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36931,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:44281,Total threads: 16
Dashboard: http://127.0.0.1:42029/status,Memory: 11.69 GiB
Nanny: tcp://127.0.0.1:40879,


In [6]:
def process_single_path(path, destination):
    """Process a single path as a synchronous Dask task"""
    try:
        if not isinstance(path, PosixPath):
            # Get the plugin from the worker
            worker = get_worker()
            plugin = worker.plugins['rustac_store']
            read_store = plugin.store
            writer_store = plugin.writer_store
            dest_path = path.replace(".ndjson", ".parquet")
        elif isinstance(path, PosixPath):
            path_parts = paths[0].parts[-2:]  # Get ('N70W030', '2015.ndjson')
            dest_path = destination.joinpath(*path_parts).with_suffix(".parquet")
            dest_path.parent.mkdir(parents=True, exist_ok=True)
            read_store = None
            writer_store = None
        else:
            raise TypeError(
                f"Expected path to be either str or PosixPath, got {type(path).__name__}"
            )
            
        value = rustac.read(str(path), store=read_store)  # Synchronous read

        if value["type"] == "Feature":
            rustac.write(str(dest_path), [value], format="parquet", store=writer_store)  # Synchronous write
        else:
            assert value["type"] == "FeatureCollection"
            rustac.write(str(dest_path), value, format="parquet", store=writer_store)  # Synchronous write

        return {'status': 'success', 'path': path}
        
    except Exception as e:
        logger.error(f"Failed to process {path}: {str(e)}")
        return {'status': 'failed', 'path': path, 'error': str(e)}

def process_files_batches_per_worker(client, paths, destination, progress=True):
    """
    Process files in batches where each batch size equals worker count.
    Ensures each worker gets only one file per batch.
    """
    n_workers = len(client.scheduler_info()['workers'])
    batches = [paths[i:i + n_workers] for i in range(0, len(paths), n_workers)]
    
    all_results = []
    for batch in batches:
        futures = [client.submit(process_single_path, path, destination) 
                  for path in batch]
        batch_results = client.gather(futures)  # Wait for current batch
        all_results.extend(batch_results)
        
        if progress:
            print(f"Processed {len(all_results)}/{len(paths)} files")
    
    return all_results

### Cloud Listing

In [ ]:
# year_file_re = re.compile(r'.*/(\d{4})\.ndjson$')

# source_store = S3Store(
#     bucket="its-live-data", prefix="test-space/cloud-experiments/catalog/sentinel1-consolidated/", region="us-west-2", skip_signature=True
# )

# paths = []
# sizes = []
# for list_stream in source_store.list():
#     for object_meta in list_stream:
#         if year_file_re.match(object_meta["path"]):
#             paths.append(object_meta["path"])
#             sizes.append(object_meta["size"])
# print(len(paths))

### Local Listing

In [5]:
import os.path


year_file_re = re.compile(r'.*/(\d{4})\.ndjson$')
paths = []
sizes = []

source = Path("../../catalog/sentinel2")
destination = Path("../../catalog/geoparquet/sentinel2")

try:
    if not destination.exists():
        destination.mkdir(parents=True, exist_ok=True)
except Exception as e:
    raise RuntimeError(f"Failed to create directory {destination}: {str(e)}")

for path in source.glob("**/*.ndjson"):
    if year_file_re.match(str(path)):
        paths.append(path)
        sizes.append(os.path.getsize(path))

print(len(paths))

1700


In [ ]:
paths[0]

In [ ]:
results = process_files_with_async_batches(
    client, 
    paths, 
    destination, 
    batch_size=280,  # Paths per worker
    semaphore_limit=4,  # Concurrent async ops per worker
    progress=False
)

In [ ]:
failed = [r["path"] for f in results]
len(failed)

In [ ]:
for r in results:
    if r["status"] != "success":
        print(r)

In [ ]:
import duckdb


destination = Path("./data/geoparquet/new")

duckdb.sql(f"select count(*) from read_parquet('{destination}/**/*.parquet')")

In [ ]:
import os.path
import humanize

destination = Path("./data/geoparquet/new")

count = 0
size = 0
for path in destination.glob("**/*.parquet"):
    count += 1
    size += os.path.getsize(path)

print(f"The {count} stac-geoparquet files are {humanize.naturalsize(size)}")
print(f"That's {100 * size / sum(sizes):.2f}% of the original size")

In [ ]:
import os.path
import humanize

destination = Path("./data/geoparquet/sentinel1")

count = 0
size = 0
for path in destination.glob("**/*.parquet"):
    count += 1
    size += os.path.getsize(path)

print(f"The {count} stac-geoparquet files are {humanize.naturalsize(size)}")
print(f"That's {100 * size / sum(sizes):.2f}% of the original size")

In [ ]:
# Debug what's happening with path parts
from pathlib import Path
from urllib.parse import urlparse

# Example path
path = "s3://its-live-data/test-space/stac-catalog/landsatOLI/v02/N30E100/2013.ndjson"

url_path = urlparse(path).path
print(f"url_path: {url_path}")

source_path = Path(url_path)
print(f"source_path.parts: {source_path.parts}")
print(f"Last 4 parts: {source_path.parts[-4:]}")

preserved_path = Path(*source_path.parts[-4:])
print(f"preserved_path: {preserved_path}")

# The fix - skip the root '/' by filtering it out
non_root_parts = [part for part in source_path.parts if part != '/']
print(f"non_root_parts: {non_root_parts}")
print(f"Last 4 non-root parts: {non_root_parts[-4:]}")

preserved_path_fixed = Path(*non_root_parts[-4:])
print(f"preserved_path_fixed: {preserved_path_fixed}")

In [ ]:


paths = [
    'landsatOLI/v02/N20E080/1987.ndjson',
    'landsatOLI/v02/N20E080/',
    'landsatOLI/v02/N20E080/README.txt',
    'landsatOLI/v02/S10W100/2003.ndjson'
]

filtered = [p for p in paths if year_file_re.match(p)]
print(filtered)